### **Stacking Ensemble**
Stacking (Stacked Generalization) is an ensemble learning technique that combines multiple base models to produce a more accurate prediction. Unlike Bagging and Boosting, which typically use homogeneous weak learners, Stacking often utilizes heterogeneous base models.

The architecture consists of:
1. **Base-Models (Level-0):** A set of diverse models are trained on the original training data.
2. **Meta-Model (Level-1):** A final model that takes the predictions of the base-models as input features and learns how to best combine them to minimize the final error.

The key idea is that the meta-model can identify when certain base models perform better than others, effectively leveraging the strengths of each individual algorithm.

In [1]:
# Importing necessary libraries
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

df = load_breast_cancer(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(df["data"], df["target"], test_size=0.2, random_state=42)
X_train.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
68,9.029,17.33,58.79,250.5,0.10660,0.14130,0.31300,0.04375,0.2111,0.08046,...,10.31,22.65,65.50,324.7,0.14820,0.43650,1.25200,0.17500,0.4228,0.11750
181,21.090,26.57,142.70,1311.0,0.11410,0.28320,0.24870,0.14960,0.2395,0.07398,...,26.68,33.48,176.50,2089.0,0.14910,0.75840,0.67800,0.29030,0.4098,0.12840
63,9.173,13.86,59.20,260.9,0.07721,0.08751,0.05988,0.02180,0.2341,0.06963,...,10.01,19.23,65.59,310.1,0.09836,0.16780,0.13970,0.05087,0.3282,0.08490
248,10.650,25.22,68.01,347.0,0.09657,0.07234,0.02379,0.01615,0.1897,0.06329,...,12.25,35.19,77.98,455.7,0.14990,0.13980,0.11250,0.06136,0.3409,0.08147
60,10.170,14.88,64.55,311.9,0.11340,0.08061,0.01084,0.01290,0.2743,0.06960,...,11.02,17.45,69.86,368.6,0.12750,0.09866,0.02168,0.02579,0.3557,0.08020


The `cv` hyperparameter in `StackingClassifier` controls **how the training data is split to train the base models and the meta-model**. It's crucial for preventing **data leakage**—ensuring the meta-model is trained on predictions from base models that haven't seen that specific data during training.

Here are all possible values, explained from simplest to most advanced.

#### **1. Integer Value (e.g., `cv=5`)**
This is the **most common and recommended** approach for most situations.

*   **Simple Analogy**: K-Fold Cross-Validation for the meta-learner's training data.
*   **How it Works**:
    1.  Your training data is split into **`k` equal folds** (e.g., 5 folds).
    2.  Each base model is trained **`k` times**. For each training round, `k-1` folds are used to train the base model, and it makes predictions on the **remaining 1 fold** it hasn't seen.
    3.  After looping through all folds, you get **out-of-fold predictions** for *every* training sample. These predictions are stacked to form the **new feature set** for the meta-model.
    4.  Finally, all base models are **retrained on the full original training set**, and the meta-model is trained on the out-of-fold predictions.

*   **Why it's good**: Maximizes data use, provides robust out-of-fold predictions, and is the standard default.

#### **2. A CV Splitter Object (e.g., `cv=StratifiedKFold(n_splits=5)`)**
This gives you **fine-grained control** over the cross-validation strategy.

*   **Common Splitters & When to Use Them**:
    *   **`StratifiedKFold`**: **Use for classification with imbalanced classes**. It preserves the percentage of each target class in every fold. This is often better than plain `KFold` for classification tasks.
    *   **`TimeSeriesSplit`**: **Use for time-series data**. It prevents future data from being used to predict the past, respecting temporal order.
    *   **`GroupKFold`**: **Use when you have grouped data** (e.g., multiple samples from the same patient). It ensures all samples from the same group are either all in the training fold or all in the validation fold, preventing data leakage.

*   **Example**:
    ```python
    from sklearn.ensemble import StackingClassifier
    from sklearn.model_selection import StratifiedKFold

    cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    stack_clf = StackingClassifier(estimators=base_learners,
                                   final_estimator=meta_learner,
                                   cv=cv_strategy)  # Use custom CV strategy
    ```

#### **3. `cv='prefit'` (The Advanced Option)**
This is a special mode where you provide **already pre-trained base models**.

*   **Simple Analogy**: The stacking ensemble acts only as a "smart voter" using ready-made experts. It does **not** retrain the base models.
*   **How it Works**:
    1.  You must **pre-fit all your base models** on the *entire* training data beforehand.
    2.  The StackingClassifier **skips the cross-validation loop**. It simply uses these pre-trained base models to make predictions on the training data.
    3.  These predictions are used directly to train the meta-model.
*   **Crucial Caveat & When to Use**:
    *   ⚠️ **Major Risk**: This leads to **severe data leakage** if you then train the meta-model on the same data used for the base models. The base models have already seen all the data, so their predictions are not "out-of-fold." The meta-model will **overfit terribly**.
    *   **Correct Use Case**: You **must provide a separate dataset** (`X`, `y`) to the `.fit()` method that the base models have **never seen before**. This is typically used in complex multi-stage training pipelines or when base models are very expensive to train.

    ```python
    # Example of CORRECT usage with 'prefit'
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier, StackingClassifier

    # 1. Create a clean hold-out set that base models NEVER see during their training
    X_full, y_full = make_classification(n_samples=1000)
    X_base, X_meta, y_base, y_meta = train_test_split(X_full, y_full, test_size=0.3, random_state=42)

    # 2. Pre-fit base models on the BASE set only
    rf = RandomForestClassifier().fit(X_base, y_base)
    lr = LogisticRegression().fit(X_base, y_base)

    # 3. Create stacking with 'prefit', and train it on the held-out META set
    stack_clf = StackingClassifier(
        estimators=[('rf', rf), ('lr', lr)],  # Provide pre-fit models
        final_estimator=LogisticRegression(),
        cv='prefit'
    )
    stack_clf.fit(X_meta, y_meta)  # Train meta-learner on unseen data
    ```

#### **4. `cv=None` (The Deprecated/Dangerous Option)**
This option is **officially deprecated** in scikit-learn (since v1.5) for good reason.

*   **What it did**: It used the **same data** to train base models and the meta-model without any cross-validation.
*   **Why it's bad**: This causes **complete data leakage**. The meta-model learns from predictions where base models have already seen the answers, leading to **extreme overfitting** and unrealistic performance estimates. **Never use this.**

#### **Summary & Quick Decision Flowchart**

| `cv` Value | Best For | Data Leakage Risk | Key Action |
| :--- | :--- | :--- | :--- |
| **Integer (e.g., `5`)** | **Most common cases**, general-purpose stacking. | **None** (Properly prevented). | Uses K-Fold CV to generate clean out-of-fold predictions. |
| **CV Splitter Object** | **Special data structures** (imbalanced, time-series, groups). | **None**. | Provides specialized folding logic for your data type. |
| **`'prefit'`** | **Advanced pipelines** with expensive pre-trained base models. | **Very High** if misused. | You must supply a **separate, unseen dataset** to `stack_clf.fit()`. |
| **`None`** | **❌ Do not use.** (Deprecated) | **Extreme.** | ⚠️ Will cause severe overfitting. |

**Your Decision Path**:
1.  **Start with `cv=5` or `cv=StratifiedKFold(5)`**. This is the safe, standard choice.
2.  Only use a **custom CV splitter** if your data has a specific structure (time series, groups, severe class imbalance).
3.  Use **`cv='prefit'`** only in very specific, advanced scenarios where you fully understand and control the separate data flow to prevent leakage. For 99% of use cases, stick with option 1.

This setup ensures your stacking ensemble learns genuine patterns from the base models' unbiased predictions, leading to robust generalization on new data.

In [2]:
# Base Estimators
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier # Meta Learner
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

# Stacking Classifier
model = StackingClassifier(
    estimators=[
        ('dt', DecisionTreeClassifier(max_depth=20, max_features='sqrt', random_state=42)),
        ('lr', LogisticRegression(max_iter=5000, random_state=42)),
        ('rf', RandomForestClassifier(max_samples=0.75, max_features='sqrt')),
    ],
    final_estimator=KNeighborsClassifier(n_neighbors=5),
    cv=5,
    passthrough=False, # When False, only the predictions of estimators will be used as training data for final_estimator. When True, the final_estimator is trained on the predictions as well as the original training data.
    verbose=True
)

# Model Training
model.fit(X_train, y_train)

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    3.4s finished
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    1.2s finished


,estimators,"[('dt', ...), ('lr', ...), ...]"
,final_estimator,KNeighborsClassifier()
,cv,5
,stack_method,'auto'
,n_jobs,None
,passthrough,False
,verbose,True
,criterion,'gini'
,splitter,'best'
,max_depth,20
,min_samples_split,2


In [3]:
# Model Accuracy
model.score(X_test, y_test)

0.9649122807017544

In [4]:
# Classification Report
from sklearn.metrics import classification_report
print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      0.91      0.95        43
           1       0.95      1.00      0.97        71

    accuracy                           0.96       114
   macro avg       0.97      0.95      0.96       114
weighted avg       0.97      0.96      0.96       114



In [5]:
# Training meta learning along with input features -> passthrough = True
model = StackingClassifier(
    estimators=[
        ('dt', DecisionTreeClassifier(max_depth=20, max_features='sqrt', random_state=42)),
        ('lr', LogisticRegression(max_iter=5000, random_state=42)),
        ('rf', RandomForestClassifier(max_samples=0.75, max_features='sqrt')),
    ],
    final_estimator=KNeighborsClassifier(n_neighbors=5),
    cv=5,
    passthrough=True,
    verbose=True
)
model.fit(X_train, y_train)

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    5.5s finished
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    1.4s finished


,estimators,"[('dt', ...), ('lr', ...), ...]"
,final_estimator,KNeighborsClassifier()
,cv,5
,stack_method,'auto'
,n_jobs,None
,passthrough,True
,verbose,True
,criterion,'gini'
,splitter,'best'
,max_depth,20
,min_samples_split,2


In [6]:
# Model Accuracy
model.score(X_test, y_test)

0.956140350877193

In [7]:
# Classification Report
print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      0.88      0.94        43
           1       0.93      1.00      0.97        71

    accuracy                           0.96       114
   macro avg       0.97      0.94      0.95       114
weighted avg       0.96      0.96      0.96       114

